In [69]:
import delta

#import
import ConnectionConfig as cc
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql import SparkSession
debugging_mode=True

In [70]:
#config

cc.setupEnvironment()
spark = cc.startLocalCluster("mongodbsetup",4)
spark.getActiveSession()

Environment variables are set...


In [71]:
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [72]:
#treusure tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("treasure")

#treusure tabel
df_stage = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stage") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_stage.createOrReplaceTempView("stage")

#treusure tabel
df_treasure_stage = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_stages") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure_stage.createOrReplaceTempView("treasure_stages")

#city tabel
df_city = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "city") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_city.createOrReplaceTempView("city")

#country tabel
df_country = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "country") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_country.createOrReplaceTempView("country")


In [73]:
treasure_mongoDb = spark.sql("""
                             SELECT t.id,
                                    t.difficulty,
                                    t.terrain,
                                    t.owner_id,
                                    co.name,
                                    c.city_name,
                                    collect_list(
                                            named_struct(
                                                    'latitude', c.latitude,
                                                    'longitude', c.longitude
                                            )
                                    ) AS city_coordinates,
                                    collect_list(
                                            named_struct(
                                                    'stage_id', st.id,
                                                    'container_size', st.container_size,
                                                    'description', st.description,
                                                    'latitude', st.latitude,
                                                    'longitude', st.longitude,
                                                    'sequence_number', st.sequence_number,
                                                    'type', st.type,
                                                    'visibility', st.visibility
                                            )
                                    ) AS stages
                             FROM treasure t
                                      JOIN city c ON t.city_city_id = c.city_id
                                      JOIN country co ON c.country_code = co.code
                                      LEFT JOIN treasure_stages ts ON t.id = ts.treasure_id
                                      LEFT JOIN stage st ON ts.stages_id = st.id
                             GROUP BY t.id, t.difficulty, t.terrain, t.owner_id,
                                      c.city_id, c.city_name, c.latitude, c.longitude, c.postal_code, c.country_code,
                                      co.code3, co.name LIMIT 100
                             """)

treasure_mongoDb.createOrReplaceTempView("treasure_mongoDb")


spark.sql("SELECT * FROM treasure_mongoDb").show()

+--------------------+----------+-------+--------------------+--------------------+-----------------+--------------------+--------------------+
|                  id|difficulty|terrain|            owner_id|                name|        city_name|    city_coordinates|              stages|
+--------------------+----------+-------+--------------------+--------------------+-----------------+--------------------+--------------------+
|[00 03 72 3C 7C C...|         3|      2|[75 AA F7 4A 33 C...|               India|     Sarandapalli|[{12.5592, 77.931...|[{[1B 32 E8 9A A6...|
|[00 05 1E A2 05 8...|         0|      2|[DD AD C2 FE 29 5...|               India|            Rajod|[{26.8438, 74.0068}]|[{[6B 7D 70 BA 78...|
|[00 08 A9 BF BE 4...|         3|      2|[1C E4 7F 2B 5F 2...|               India|        Indrapala|[{22.7771, 88.1942}]|[{[3B 60 7F 16 34...|
|[00 08 B9 20 1B F...|         1|      2|[18 61 40 54 61 3...|United States of ...|            Dover|[{27.992, -82.2138}]|[{[13 5B B8 C2

In [74]:
treasure_mongoDb.write \
    .format("json") \
    .mode("overwrite") \
    .save("treasure_export.json")